# 5.1 LangSmith — RAG 평가 (Evaluation, Pinecone)

RAG 챗봇(RagBot)을 만들고 **LangSmith로 평가**한다. 한 노트북에 두 작업:
- 앞부분(1~6): retriever + RagBot (raw `openai` SDK + `wrap_openai`/`@traceable`)
- 뒷부분(7~10): 평가셋 생성 → predict → Evaluator 3종 → `evaluate()`
- 바꿔 쓸 곳: `.env`의 키, `index_name`, `dataset_name`

# 1. 패키지 설치

In [1]:
%pip install -U langchain langchain-openai langchain-pinecone langsmith openai python-dotenv docx2txt

  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.8.11
    Uninstalling langsmith-0.8.11:
      Successfully uninstalled langsmith-0.8.11
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# 2. 환경 변수

- `.env`에 `OPENAI_API_KEY`, `PINECONE_API_KEY` 필요
- 평가 결과를 LangSmith에 올리려면 `LANGSMITH_TRACING=true`, `LANGSMITH_API_KEY` 설정

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

# 3. Pinecone 인덱스 구축 (최초 1회)

- `tax_with_markdown.docx` → chunking → 임베딩 → 서버리스 인덱스
- `text-embedding-3-large` → **dimension 3072**, `metric='cosine'`
- 인덱스가 없으면 생성하고, **비어 있을 때만** 문서를 업로드 (`total_vector_count == 0` 가드 → 재실행 시 건너뜀)

In [ ]:
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

index_name = 'tax-index'
embedding = OpenAIEmbeddings(model='text-embedding-3-large')  # dimension 3072

pinecone = Pinecone()

# 인덱스가 없으면 생성 (최초 1회)
if not pinecone.has_index(index_name):
    pinecone.create_index(
        name=index_name,
        dimension=3072,  # text-embedding-3-large
        metric='cosine',
        spec=ServerlessSpec(cloud='aws', region='us-east-1'),
    )

# 인덱스가 비어 있을 때만 문서 임베딩·업로드 (최초 1회 → 재실행 시 비용/중복 방지)
if pinecone.Index(index_name).describe_index_stats()['total_vector_count'] == 0:
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500,
        chunk_overlap=200,
    )
    loader = Docx2txtLoader('./tax_with_markdown.docx')
    document_list = loader.load_and_split(text_splitter=text_splitter)
    PineconeVectorStore.from_documents(
        document_list,
        embedding,
        index_name=index_name,
    )

# 4. Retriever

- 이미 구축된 인덱스에 연결 → `as_retriever()` (기본 `k=4`)

In [5]:
database = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

retriever = database.as_retriever()

# 5. RagBot

- `wrap_openai(openai.Client())` + `@traceable()`: LangSmith 자동 트레이싱
- `get_answer` → `retrieve_docs` → `invoke_llm` (LangChain 체인 없이 직접 구현)
- `invoke_llm`은 `{answer, contexts}` 반환 (contexts는 hallucination 평가용)

In [6]:
import openai
from langsmith import traceable
from langsmith.wrappers import wrap_openai


class RagBot:
    def __init__(self, retriever, model: str = "gpt-4o"):
        self._retriever = retriever
        # LangSmith가 OpenAI 호출을 자동 트레이싱하도록 client를 래핑
        self._client = wrap_openai(openai.Client())
        self._model = model

    @traceable()
    def retrieve_docs(self, question):
        return self._retriever.invoke(question)

    @traceable()
    def invoke_llm(self, question, docs):
        # retrieve_docs()로 가져온 문서들을 system prompt로 전달 (3.3 방식과 유사)
        response = self._client.chat.completions.create(
            model=self._model,
            messages=[
                {
                    "role": "system",
                    "content": "당신은 한국의 소득세 전문가입니다. "
                    "아래 소득세법을 참고해서 사용자의 질문에 답변해주세요.\n\n"
                    f"## 소득세법\n\n{docs}",
                },
                {"role": "user", "content": question},
            ],
        )

        # Evaluator에서 answer와 context를 평가할 예정
        return {
            "answer": response.choices[0].message.content,
            "contexts": [str(doc) for doc in docs],
        }

    @traceable()
    def get_answer(self, question: str):
        docs = self.retrieve_docs(question)
        return self.invoke_llm(question, docs)


rag_bot = RagBot(retriever)

# 6. 동작 확인

- `get_answer`는 `{answer, contexts}` dict 반환

In [7]:
answer = rag_bot.get_answer('거주자의 소득세 납세지는 어디인가요?')
answer['answer']

'거주자의 소득세 납세지는 그 주소지로 합니다. 다만, 주소지가 없는 경우에는 그 거소지로 합니다.'

# 7. 평가 데이터셋 생성

- Evaluation에 쓸 Question-Answer(+context) pair 20개
- ⚠️ 강의의 `inputs=/outputs=/metadata=` 병렬 리스트는 langsmith 0.8.x에서 위험 → **`examples=[{...}]` 신형**
- `has_dataset` 가드로 재실행 시 중복 생성 방지

In [ ]:
from langsmith import Client

client = Client()

dataset_name = "income_tax_dataset"

# 평가용 question-answer(+context) pair
# langsmith 0.8.x: 강의의 inputs=/outputs=/metadata= 병렬 리스트 -> examples=[{...}] 신형
examples = [
    {
        "inputs": {"input_question": '제1조에 따른 소득세법의 목적은 무엇인가요?'},
        "outputs": {"output_answer": '소득세법의 목적은 소득의 성격과 납세자의 부담능력에 따라 적정하게 과세함으로써 조세부담의 형평을 도모하고 재정수입의 원활한 조달에 이바지하는 것입니다.'},
        "metadata": {"contexts": '제1조(목적) 이 법은 개인의 소득에 대하여 소득의 성격과 납세자의 부담능력 등에 따라 적정하게 과세함으로써 조세부담의 형평을 도모하고 재정수입의 원활한 조달에 이바지함을 목적으로 한다.'},
    },
    {
        "inputs": {"input_question": "'거주자'는 소득세법에서 어떻게 정의되나요?"},
        "outputs": {"output_answer": "'거주자'는 한국에 주소를 두거나 183일 이상 거소를 둔 개인을 의미합니다."},
        "metadata": {"contexts": '제1조의2(정의) “거주자”란 국내에 주소를 두거나 183일 이상의 거소를 둔 개인을 말한다.'},
    },
    {
        "inputs": {"input_question": "'비거주자'는 소득세법에 따라 어떻게 정의되나요?"},
        "outputs": {"output_answer": "'비거주자'는 거주자가 아닌 개인을 의미합니다."},
        "metadata": {"contexts": '제1조의2(정의) “비거주자”란 거주자가 아닌 개인을 말한다.'},
    },
    {
        "inputs": {"input_question": "소득세법에 따른 '내국법인'은 누구를 의미하나요?"},
        "outputs": {"output_answer": "'내국법인'은 법인세법 제2조 제1호에 따른 내국법인을 의미합니다."},
        "metadata": {"contexts": '제1조의2(정의) “내국법인”이란 「법인세법」 제2조제1호에 따른 내국법인을 말한다.'},
    },
    {
        "inputs": {"input_question": '소득세법에 따라 소득세를 납부할 의무가 있는 사람은 누구인가요?'},
        "outputs": {"output_answer": '거주자 및 국내원천소득이 있는 비거주자는 소득세를 납부할 의무가 있습니다.'},
        "metadata": {"contexts": '제2조(납세의무) 거주자 및 국내원천소득이 있는 비거주자는 소득세를 납부할 의무가 있다.'},
    },
    {
        "inputs": {"input_question": '거주자의 과세 범위는 무엇인가요?'},
        "outputs": {"output_answer": '거주자는 법에서 규정한 모든 소득에 대해 과세되며, 비거주자는 국내원천소득에 대해서만 과세됩니다.'},
        "metadata": {"contexts": '제3조(과세소득의 범위) 거주자는 법에서 규정한 모든 소득에 대해 과세되며, 비거주자는 국내원천소득에 대해서만 과세된다.'},
    },
    {
        "inputs": {"input_question": '소득세법에 따라 소득은 어떻게 분류되나요?'},
        "outputs": {"output_answer": '소득은 종합소득, 퇴직소득, 양도소득으로 분류됩니다.'},
        "metadata": {"contexts": '제4조(소득의 구분) 소득은 종합소득, 퇴직소득, 양도소득으로 분류된다.'},
    },
    {
        "inputs": {"input_question": '종합소득이란 무엇인가요?'},
        "outputs": {"output_answer": '종합소득은 이자소득, 배당소득, 사업소득, 근로소득, 연금소득 및 기타소득을 포함합니다.'},
        "metadata": {"contexts": '제4조(소득의 구분) 종합소득은 이자소득, 배당소득, 사업소득, 근로소득, 연금소득 및 기타소득을 포함한다.'},
    },
    {
        "inputs": {"input_question": '세금이 면제되는 소득의 종류는 무엇인가요?'},
        "outputs": {"output_answer": '비과세 소득에는 공익신탁의 이익, 특정 사업소득 및 기타 법에서 정한 특정 소득이 포함됩니다.'},
        "metadata": {"contexts": '제12조(비과세소득) 비과세 소득에는 공익신탁의 이익, 특정 사업소득 및 기타 법에서 정한 특정 소득이 포함된다.'},
    },
    {
        "inputs": {"input_question": '소득세의 과세기간은 어떻게 되나요?'},
        "outputs": {"output_answer": '소득세의 과세기간은 매년 1월 1일부터 12월 31일까지입니다.'},
        "metadata": {"contexts": '제5조(과세기간) 소득세의 과세기간은 매년 1월 1일부터 12월 31일까지이다.'},
    },
    {
        "inputs": {"input_question": '거주자의 소득세 납세지는 어디인가요?'},
        "outputs": {"output_answer": '거주자의 소득세 납세지는 주소지이며, 주소지가 없으면 거소지입니다.'},
        "metadata": {"contexts": '제6조(납세지) 거주자의 소득세 납세지는 주소지이며, 주소지가 없으면 거소지이다.'},
    },
    {
        "inputs": {"input_question": '비거주자의 소득세 납세지는 어디인가요?'},
        "outputs": {"output_answer": '비거주자의 소득세 납세지는 국내사업장의 소재지입니다. 국내사업장이 여러 곳인 경우 주된 사업장의 소재지가 납세지가 됩니다.'},
        "metadata": {"contexts": '제6조(납세지) 비거주자의 소득세 납세지는 국내사업장의 소재지이다. 국내사업장이 여러 곳인 경우 주된 사업장의 소재지이다.'},
    },
    {
        "inputs": {"input_question": '납세지가 불분명한 경우 어떻게 되나요?'},
        "outputs": {"output_answer": '납세지가 불분명한 경우 대통령령으로 정합니다.'},
        "metadata": {"contexts": '제6조(납세지) 납세지가 불분명한 경우에는 대통령령으로 정한다.'},
    },
    {
        "inputs": {"input_question": '원천징수세액의 납세지는 어떻게 결정되나요?'},
        "outputs": {"output_answer": '원천징수세액의 납세지는 원천징수자의 종류와 위치에 따라 결정됩니다.'},
        "metadata": {"contexts": '제7조(원천징수 등의 경우의 납세지) 원천징수세액의 납세지는 원천징수자의 종류와 위치에 따라 결정된다.'},
    },
    {
        "inputs": {"input_question": '납세자의 사망 시 납세지는 어떻게 되나요?'},
        "outputs": {"output_answer": '납세자의 사망 시 상속인 또는 납세관리인의 주소지나 거소지가 납세지가 됩니다.'},
        "metadata": {"contexts": '제8조(상속 등의 경우의 납세지) 납세자의 사망 시 상속인 또는 납세관리인의 주소지나 거소지가 납세지가 된다.'},
    },
    {
        "inputs": {"input_question": '신탁 소득에 대한 납세의 범위는 무엇인가요?'},
        "outputs": {"output_answer": '신탁 소득에 대한 납세의 범위는 신탁의 수익자가 해당 소득에 대해 납세의무를 집니다.'},
        "metadata": {"contexts": '제2조의3(신탁재산 귀속 소득에 대한 납세의무의 범위) 신탁 소득에 대한 납세의 범위는 신탁의 수익자가 해당 소득에 대해 납세의무를 진다.'},
    },
    {
        "inputs": {"input_question": '원천징수 대상 소득은 무엇인가요?'},
        "outputs": {"output_answer": '이자소득, 배당소득 및 기타 법에서 정한 소득은 원천징수 대상입니다.'},
        "metadata": {"contexts": '제14조(과세표준의 계산) 이자소득, 배당소득 및 기타 법에서 정한 소득은 원천징수 대상이다.'},
    },
    {
        "inputs": {"input_question": '공동 소유 자산의 양도소득은 어떻게 과세되나요?'},
        "outputs": {"output_answer": '공동 소유 자산의 양도소득은 각 거주자 소유 지분에 따라 과세됩니다.'},
        "metadata": {"contexts": '제14조(과세표준의 계산) 공동 소유 자산의 양도소득은 각 거주자 소유 지분에 따라 과세된다.'},
    },
    {
        "inputs": {"input_question": '이자 소득의 출처는 무엇인가요?'},
        "outputs": {"output_answer": '이자 소득의 출처는 정부 및 지방자치단체가 발행한 채권, 법인이 발행한 채권, 국내외 은행 예금 등입니다.'},
        "metadata": {"contexts": '제16조(이자소득) 이자 소득의 출처는 정부 및 지방자치단체가 발행한 채권, 법인이 발행한 채권, 국내외 은행 예금 등이다.'},
    },
    {
        "inputs": {"input_question": '소득세법에서 배당소득은 어떻게 정의되나요?'},
        "outputs": {"output_answer": '배당소득은 국내외 법인으로부터 받는 배당금 및 배분금, 기타 법에서 정한 소득을 포함합니다.'},
        "metadata": {"contexts": '제17조(배당소득) 배당소득은 국내외 법인으로부터 받는 배당금 및 배분금, 기타 법에서 정한 소득을 포함한다.'},
    },
]

# 재실행 시 중복 생성 방지 (has_dataset 가드)
if client.has_dataset(dataset_name=dataset_name):
    dataset = client.read_dataset(dataset_name=dataset_name)
else:
    dataset = client.create_dataset(dataset_name)
    client.create_examples(dataset_id=dataset.id, examples=examples)

# 8. 예측 함수

- `evaluate`가 각 example을 넣어 호출 → RagBot 답변 생성
- hallucination까지 보려면 `contexts`도 반환하는 `_with_context` 사용

In [ ]:
def predict_rag_answer(example: dict):
    """답변만 평가할 때 사용"""
    response = rag_bot.get_answer(example["input_question"])
    return {"answer": response["answer"]}


def predict_rag_answer_with_context(example: dict):
    """context까지 활용해 hallucination을 평가할 때 사용"""
    response = rag_bot.get_answer(example["input_question"])
    return {"answer": response["answer"], "contexts": response["contexts"]}

# 9. Evaluator (정확도 / 도움도 / 할루시네이션)

- 강의 `from langchain import hub` + `hub.pull(...)`는 **v1에서 제거** → `client.pull_prompt("langchain-ai/...", dangerously_pull_public_prompt=True)`
- 공개 프롬프트는 `StructuredPrompt` → `prompt | llm` 하면 구조화 출력(`score["Score"]`)
- LLM Judge: `gpt-4o`, `temperature=0`

In [ ]:
from langchain_openai import ChatOpenAI

# 답변 정확도(정답 reference와 비교)
grade_prompt_answer_accuracy = client.pull_prompt(
    "langchain-ai/rag-answer-vs-reference", dangerously_pull_public_prompt=True
)


def answer_evaluator(run, example) -> dict:
    input_question = example.inputs["input_question"]
    reference = example.outputs["output_answer"]
    prediction = run.outputs["answer"]

    llm = ChatOpenAI(model="gpt-4o", temperature=0)
    answer_grader = grade_prompt_answer_accuracy | llm

    score = answer_grader.invoke({
        "question": input_question,
        "correct_answer": reference,
        "student_answer": prediction,
    })
    score = score["Score"]
    return {"key": "answer_v_reference_score", "score": score}

In [ ]:
# 답변이 질문에 얼마나 도움되는지 (정답과 비교하지 않음)
grade_prompt_answer_helpfulness = client.pull_prompt(
    "langchain-ai/rag-answer-helpfulness", dangerously_pull_public_prompt=True
)


def answer_helpfulness_evaluator(run, example) -> dict:
    input_question = example.inputs["input_question"]
    prediction = run.outputs["answer"]

    llm = ChatOpenAI(model="gpt-4o", temperature=0)
    answer_grader = grade_prompt_answer_helpfulness | llm

    score = answer_grader.invoke({
        "question": input_question,
        "student_answer": prediction,
    })
    score = score["Score"]
    return {"key": "answer_helpfulness_score", "score": score}

In [ ]:
# 검색된 context에 근거한 답변인지 (hallucination 여부)
grade_prompt_hallucinations = client.pull_prompt(
    "langchain-ai/rag-answer-hallucination", dangerously_pull_public_prompt=True
)


def answer_hallucination_evaluator(run, example) -> dict:
    input_question = example.inputs["input_question"]
    contexts = run.outputs["contexts"]
    prediction = run.outputs["answer"]

    llm = ChatOpenAI(model="gpt-4o", temperature=0)
    answer_grader = grade_prompt_hallucinations | llm

    score = answer_grader.invoke({
        "documents": contexts,
        "student_answer": prediction,
    })
    score = score["Score"]
    return {"key": "answer_hallucination", "score": score}

# 10. 평가 실행

- `evaluate` 결과는 LangSmith 대시보드에 업로드됨 (https://smith.langchain.com/)
- `predict_rag_answer_with_context`를 써서 정확도·도움도·할루시네이션 3종 동시 평가

In [ ]:
from langsmith.evaluation import evaluate

experiment_results = evaluate(
    predict_rag_answer_with_context,  # context까지 넘겨 hallucination도 평가
    data=dataset_name,
    evaluators=[answer_evaluator, answer_helpfulness_evaluator, answer_hallucination_evaluator],
    experiment_prefix="income-tax-eval",
    metadata={"version": "income tax v1, gpt-4o"},
)